# Datos agrupados: exceso de confianza y agrupamiento parcial (*partial pooling*)

Complemento de las páginas del curso *Datos agrupados y jerárquicos* y *Cómo: analizar datos agrupados*.

Mia estudia una variante génica que, para simplificar la simulación, portan familias enteras (en familias reales el genotipo también varía entre parientes, pero los parientes siguen pareciéndose entre sí), y mide el colesterol LDL en 20 parientes por familia. La variable de respuesta está en unidades arbitrarias. Los parientes de una misma familia se parecen entre sí. Simulamos esto muchas veces a partir de una verdad conocida y nos preguntamos:

1. ¿Con qué frecuencia acierta un intervalo del 95 %, y con qué frecuencia excluye el cero por error cuando **no** hay efecto, según la forma de analizar los datos?
2. ¿Cómo crece el problema cuanto más se diferencian las familias?
3. ¿Qué hace el **agrupamiento parcial** con las estimaciones de los grupos pequeños y grandes?

Cambia los parámetros de la siguiente celda y usa *Run All*. Necesita `numpy`, `pandas`, `matplotlib`, `statsmodels`.

In [ ]:
# ---- Parameters: change these and re-run ----
N_CLUSTERS = 15        # number of families (half are carrier families)
PER_CLUSTER = 20       # relatives per family
CLUSTER_SD = 3.0       # how much families differ from each other (SD of the family effect)
NOISE_SD = 3.0         # how much relatives differ within a family
TRUE_EFFECT = 0.0      # true effect of carrying the variant; 0 lets us count false positives
N_SIMS = 300           # simulated datasets per setting
SEED = 0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")   # mixed models can warn about boundary fits in some simulated datasets
rng = np.random.default_rng(SEED)


def simulate_clusters(n_clusters=N_CLUSTERS, per_cluster=PER_CLUSTER, cluster_sd=CLUSTER_SD):
    family = np.repeat(np.arange(n_clusters), per_cluster)
    carrier_family = rng.permutation(np.arange(n_clusters) % 2)     # carrier status is set per family
    x = carrier_family[family]
    y = TRUE_EFFECT * x + rng.normal(0, cluster_sd, n_clusters)[family] + rng.normal(0, NOISE_SD, len(family))
    return pd.DataFrame({"family": family, "x": x, "y": y})


def intervals(df, include_mixed=True):
    """95% CI for the carrier effect under each analysis."""
    models = {
        "ignoring clusters": smf.ols("y ~ x", df).fit(),
        "cluster-robust SE": smf.ols("y ~ x", df).fit(cov_type="cluster", cov_kwds={"groups": df["family"]}, use_t=True),
        "analysis of cluster means": smf.ols("y ~ x", df.groupby(["family", "x"], as_index=False)["y"].mean()).fit(),
    }
    if include_mixed:
        models["multilevel model"] = smf.mixedlm("y ~ x", df, groups=df["family"]).fit()
    return {name: tuple(model.conf_int().loc["x"]) for name, model in models.items()}

## 1. ¿Qué grado de confianza merece cada análisis?

Para cada estudio simulado calculamos el intervalo del 95 % del efecto de ser portador de cuatro maneras. Un método fiable tiene una **cobertura cercana al 95 %** y, cuando el efecto verdadero es 0, **excluye el 0 en aproximadamente el 5 % de los estudios**.

In [ ]:
records = []
for _ in range(N_SIMS):
    for name, (low, high) in intervals(simulate_clusters()).items():
        records.append({"method": name, "covers truth": low <= TRUE_EFFECT <= high,
                        "excludes 0": not (low <= 0 <= high), "width": high - low})
summary = pd.DataFrame(records).groupby("method").mean().round(2)
print(f"True effect = {TRUE_EFFECT}; {N_CLUSTERS} families x {PER_CLUSTER} relatives; {N_SIMS} simulated studies")
summary.loc[["ignoring clusters", "cluster-robust SE", "multilevel model", "analysis of cluster means"]]

## 2. El problema crece cuanto más se diferencian las familias

El mismo estudio, pero variando `CLUSTER_SD`. Cuando las familias son idénticas (desviación típica 0), la agrupación es irrelevante; cuanto más difieren las familias, más sobrestima su precisión un análisis ingenuo.

In [ ]:
rows = []
for sd in [0, 1, 2, 3, 5]:
    excluded = {"ignoring clusters": [], "cluster-robust SE": [], "analysis of cluster means": []}
    for _ in range(N_SIMS):
        for name, (low, high) in intervals(simulate_clusters(cluster_sd=sd), include_mixed=False).items():
            excluded[name].append(not (low <= 0 <= high))
    icc = sd**2 / (sd**2 + NOISE_SD**2)
    rows.append({"CLUSTER_SD": sd, "ICC": round(icc, 2), **{k: round(np.mean(v), 2) for k, v in excluded.items()}})
print("Share of studies whose interval excludes 0 (should be about 0.05 when the true effect is 0):")
pd.DataFrame(rows).set_index("CLUSTER_SD")

## 3. Agrupamiento parcial: los grupos pequeños se acercan a la media

Ahora, 25 familias de tamaños muy desiguales (de 2 a 40 parientes). La media *bruta* de cada familia es la estimación sin agrupamiento; la estimación del modelo multinivel acerca la media de cada familia hacia la media global, más aún en las familias pequeñas y ruidosas. Aquí conocemos la media verdadera de cada familia, así que podemos comprobar qué estimación se acerca más.

In [ ]:
n_families = 25


def draw_families():
    """One simulated set of families of very uneven size, with raw and partially pooled family means."""
    sizes = rng.integers(2, 41, n_families)
    true_means = 20 + rng.normal(0, CLUSTER_SD, n_families)
    family = np.repeat(np.arange(n_families), sizes)
    data = pd.DataFrame({"family": family, "y": true_means[family] + rng.normal(0, NOISE_SD, len(family))})
    raw = data.groupby("family")["y"].mean().to_numpy()
    fit = smf.mixedlm("y ~ 1", data, groups=data["family"]).fit()
    shrunk = np.array([fit.fe_params["Intercept"] + fit.random_effects[g].iloc[0] for g in range(n_families)])
    return sizes, true_means, raw, shrunk, data["y"].mean()


# Average the comparison over many simulated sets of families: a single draw can go either way.
errors = []
for _ in range(N_SIMS // 3):
    _, truth, raw_means, shrunk_means, _ = draw_families()
    errors.append((np.mean((raw_means - truth) ** 2), np.mean((shrunk_means - truth) ** 2)))
errors = np.array(errors)
print(f"RMSE vs. the true family means, averaged over {len(errors)} simulations:")
print(f"  no pooling (raw family means): {np.sqrt(errors[:, 0].mean()):.2f}")
print(f"  partial pooling:               {np.sqrt(errors[:, 1].mean()):.2f}")

sizes, true_means, raw, shrunk, overall = draw_families()    # one more draw, for the plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sizes, raw - overall, label="raw family mean (no pooling)")
ax.scatter(sizes, shrunk - overall, marker="x", label="partial pooling")
ax.axhline(0, color="gray", linestyle=":")
ax.set_xlabel("relatives measured in the family")
ax.set_ylabel("estimate minus the overall mean")
ax.legend()
plt.show()

## Cosas para probar

- Pon `CLUSTER_SD = 0`: los cuatro métodos se portan bien, porque no hay agrupación que ignorar.
- Pon `N_CLUSTERS = 6`: los métodos honestos siguen siendo válidos, pero sus intervalos se vuelven muy anchos (unas 10 unidades, frente a unas 6 con 15 familias), porque la muestra efectiva son seis familias.
- Pon `N_CLUSTERS = 60`: los intervalos honestos se estrechan, pero ignorar los grupos sigue excluyendo el 0 en aproximadamente la mitad de los estudios. El problema no desaparece con más familias; es una propiedad del análisis.
- Pon `TRUE_EFFECT = 2`: la columna `excludes 0` pasa a ser la potencia para detectar el efecto. El análisis ingenuo parece mucho más potente (alrededor del 70 %) que los honestos (alrededor del 20 %), y la diferencia es falsa confianza.
- En la sección 3, los puntos de las familias pequeñas son los que más se desplazan hacia cero, y el RMSE del agrupamiento parcial debería ser menor que el de las medias brutas. Aumenta `NOISE_SD` y el efecto del agrupamiento se hará más fuerte.